In [ ]:
import pandas as pd
import numpy as np
import random
from datetime import timedelta
# =========================================================
# ASSUMPTIONS
# =========================================================
# Existing DataFrames:
#
df_suppliers = pd.read_csv("data/raw/df_suppliers.csv")
df_po=pd.read_csv("data/raw/df_purchase_orders.csv")
df_shipments=pd.read_csv("data/raw/df_shipments.csv")
df_quality_inspections=pd.read_csv("data/raw/df_quality_inspections.csv")
df_inventory=pd.read_csv("data/raw/df_inventory.csv")
df_supplier_audits=pd.read_csv("data/raw/df_supplier_audits.csv")
#
# =========================================================


incident_rows = []

incident_counter = 1


# =========================================================
# BUILD SUPPLIER RISK PROFILE
# =========================================================

# ---------------------------------------------------------
# Shipment risk
# ---------------------------------------------------------

shipment_metrics = (

    df_shipments

    .groupby("supplier_id")

    .agg(

        total_shipments=(
            "shipment_id",
            "count"
        ),

        delayed_shipments=(
            "shipment_status",
            lambda x:
                (x == "Delayed").sum()
        )

    )

    .reset_index()

)

shipment_metrics["delay_ratio"] = (

    shipment_metrics["delayed_shipments"]

    /

    shipment_metrics["total_shipments"]

)


# ---------------------------------------------------------
# Quality risk
# ---------------------------------------------------------

quality_metrics = (

    df_quality_inspections

    .groupby("supplier_id")

    .agg(

        total_inspected=(
            "inspected_units",
            "sum"
        ),

        total_rejected=(
            "rejected_units",
            "sum"
        )

    )

    .reset_index()

)

quality_metrics["rejection_ratio"] = (

    quality_metrics["total_rejected"]

    /

    quality_metrics["total_inspected"]

)

quality_metrics["rejection_ratio"] = (

    quality_metrics["rejection_ratio"]

    .fillna(0)

)


# ---------------------------------------------------------
# Inventory instability
# ---------------------------------------------------------

inventory_metrics = (

    df_inventory

    .groupby("supplier_id")

    .apply(

        lambda x:

        (
            x["current_stock"]

            <

            x["safety_stock"]

        ).mean()

    )

    .reset_index(name="low_stock_ratio")

)


# ---------------------------------------------------------
# Audit risk
# ---------------------------------------------------------

audit_metrics = (

    df_supplier_audits

    .groupby("supplier_id")

    .agg(

        avg_audit_score=(
            "audit_score",
            "mean"
        )

    )

    .reset_index()

)


# =========================================================
# MERGE RISK PROFILE
# =========================================================

supplier_risk = (

    df_suppliers[[
        "supplier_id",
        "criticality_level",
        "country"
    ]]

    .merge(
        shipment_metrics,
        on="supplier_id",
        how="left"
    )

    .merge(
        quality_metrics,
        on="supplier_id",
        how="left"
    )

    .merge(
        inventory_metrics,
        on="supplier_id",
        how="left"
    )

    .merge(
        audit_metrics,
        on="supplier_id",
        how="left"
    )

)

supplier_risk = supplier_risk.fillna(0)


# =========================================================
# INCIDENT TYPES
# =========================================================

incident_types = {

    "logistics": [
        "Shipment Delay",
        "Port Congestion",
        "Customs Hold",
        "Carrier Failure"
    ],

    "quality": [
        "Quality Failure",
        "Batch Rejection",
        "Product Recall"
    ],

    "inventory": [
        "Stockout",
        "Supply Shortage"
    ],

    "compliance": [
        "Audit Non-Compliance",
        "Regulatory Violation"
    ],

    "operations": [
        "Production Shutdown",
        "Labor Strike",
        "Cybersecurity Incident"
    ]
}


# =========================================================
# GENERATE INCIDENTS
# =========================================================

for _, supplier in supplier_risk.iterrows():

    supplier_id = supplier["supplier_id"]

    criticality = supplier["criticality_level"]


    # -----------------------------------------------------
    # BASE INCIDENT PROBABILITY
    # -----------------------------------------------------

    incident_probability = 0.05


    incident_probability += (

        supplier["delay_ratio"] * 0.40

    )

    incident_probability += (

        supplier["rejection_ratio"] * 0.80

    )

    incident_probability += (

        supplier["low_stock_ratio"] * 0.30

    )


    if supplier["avg_audit_score"] < 70:

        incident_probability += 0.25

    elif supplier["avg_audit_score"] < 85:

        incident_probability += 0.10


    if criticality >= 4:

        incident_probability += 0.10


    incident_probability = min(
        incident_probability,
        0.95
    )


    # -----------------------------------------------------
    # INCIDENT COUNT
    # -----------------------------------------------------

    incident_count = np.random.poisson(

        incident_probability * 4

    )


    for _ in range(incident_count):

        # -------------------------------------------------
        # INCIDENT CATEGORY
        # -------------------------------------------------

        risk_weights = {

            "logistics":
                supplier["delay_ratio"],

            "quality":
                supplier["rejection_ratio"],

            "inventory":
                supplier["low_stock_ratio"],

            "compliance":
                max(
                    0,
                    (80 - supplier["avg_audit_score"]) / 100
                ),

            "operations":
                0.10
        }


        total_weight = sum(
            risk_weights.values()
        )


        if total_weight == 0:

            incident_category = "operations"

        else:

            normalized_weights = [

                v / total_weight

                for v in risk_weights.values()
            ]


            incident_category = np.random.choice(

                list(risk_weights.keys()),

                p=normalized_weights
            )


        # -------------------------------------------------
        # INCIDENT TYPE
        # -------------------------------------------------

        incident_type = random.choice(

            incident_types[incident_category]

        )


        # -------------------------------------------------
        # SEVERITY
        # -------------------------------------------------

        severity_weights = {

            "Low": 0.40,
            "Medium": 0.35,
            "High": 0.20,
            "Critical": 0.05
        }


        # Worse suppliers more severe incidents
        if supplier["avg_audit_score"] < 70:

            severity_weights["Critical"] += 0.10

            severity_weights["High"] += 0.10


        severity = np.random.choice(

            list(severity_weights.keys()),

            p=np.array(
                list(severity_weights.values())
            ) / sum(severity_weights.values())

        )


        # -------------------------------------------------
        # OPERATIONAL IMPACT
        # -------------------------------------------------

        impact_ranges = {

            "Low": (1, 8),

            "Medium": (8, 24),

            "High": (24, 72),

            "Critical": (72, 240)

        }


        low, high = impact_ranges[severity]

        operational_impact_hours = round(

            np.random.uniform(low, high),

            2
        )


        # -------------------------------------------------
        # INCIDENT DATE
        # -------------------------------------------------

        incident_date = (

            pd.Timestamp("2025-01-01")

            +

            timedelta(
                days=np.random.randint(0, 365)
            )

        )


        # -------------------------------------------------
        # CREATE RECORD
        # -------------------------------------------------

        incident_rows.append({

            "incident_id":
                f"INC{incident_counter:07}",

            "supplier_id":
                supplier_id,

            "incident_date":
                incident_date.date(),

            "incident_type":
                incident_type,

            "severity":
                severity,

            "operational_impact_hours":
                operational_impact_hours
        })


        incident_counter += 1


# =========================================================
# FINAL INCIDENT TABLE
# =========================================================

df_supplier_incidents = pd.DataFrame(
    incident_rows
)


# =========================================================
# OPTIONAL VALIDATION
# =========================================================

print("\n==============================")
print("INCIDENT SUMMARY")
print("==============================")

print(
    f"Total Incidents: "
    f"{len(df_supplier_incidents)}"
)

print("\nSeverity Distribution:")

print(
    df_supplier_incidents[
        "severity"
    ].value_counts()
)

print("\nIncident Types:")

print(
    df_supplier_incidents[
        "incident_type"
    ].value_counts()
)

print("\nImpact Hours Summary:")

print(
    df_supplier_incidents[
        "operational_impact_hours"
    ].describe()
)

print("\nSample Records:\n")

print(
    df_supplier_incidents.head()
)

In [ ]:
df_supplier_incidents.to_csv("data/raw/df_supplier_incidents.csv", index=False)